# Parsel: train on a Colab GPU

Trains any config in `configs/train/` on OpenEarthMap. Set `CONFIG` (and optionally `EXTRA` overrides) in the Drive cell, then run all. Each experiment gets its own run folder on Drive, so change `RUN_NAME` for ablations.

1. **Runtime > Change runtime type > GPU** (T4 is enough), then **Run all**.
2. Approve the Google Drive prompt when it appears: checkpoints and logs are written to `MyDrive/parsel-runs`, so they survive disconnects.
3. If the session disconnects, reconnect and run all again. Data setup is skipped when the tiles exist, and training resumes from `last.pt`.

Hyperparameters come from the chosen config. Only `num_workers` is lowered for Colab's 2 CPUs, and mixed precision (fp16) switches on automatically on CUDA.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

## Drive (for checkpoints)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/parsel-runs"
# Any file in configs/train/: unet_r34, segformer_b2, upernet_convnext_t, deeplabv3plus_r50
CONFIG = "unet_r34"
# Config overrides for ablations, e.g. "--set loss.dice_weight=0 --set data.crop_size=256"
EXTRA = ""
RUN_NAME = f"{CONFIG}_colab"

## Code and dependencies

In [ ]:
!test -d /content/parsel-seg || git clone -q https://github.com/barisparlakk/parsel-seg.git /content/parsel-seg
%cd /content/parsel-seg
!git pull -q
!pip install -q segmentation-models-pytorch==0.5.0 albumentations==2.0.8 rasterio pyyaml tensorboard
!pip install -q -e . --no-deps --ignore-requires-python

## Data

Downloads OpenEarthMap (9.1 GB), remaps the classes and cuts 512 px tiles. Takes roughly 10 to 20 minutes; skipped if the tiles already exist.

In [ ]:
%%bash
set -e
cd /content/parsel-seg
if [ -f data/processed/openearthmap/manifest.csv ]; then echo "tiles already built"; exit 0; fi
python scripts/download_data.py --dataset openearthmap
python scripts/remap_masks.py --dataset openearthmap
python scripts/make_tiles.py --dataset openearthmap --tile-size 512
rm -f data/raw/openearthmap/OpenEarthMap.zip
du -sh data/processed/openearthmap

## Train

Resumes automatically if a checkpoint for this run already exists on Drive.

In [ ]:
import os

last = f"{RUNS}/{RUN_NAME}/last.pt"
resume = f"--resume {last}" if os.path.exists(last) else ""
print("resuming from", last if resume else "scratch")
!python scripts/train.py --config configs/train/{CONFIG}.yaml --name {RUN_NAME} --runs {RUNS} --set data.num_workers=2 {EXTRA} {resume}

## Evaluate on the held-out test regions

In [ ]:
!python scripts/evaluate.py --run {RUNS}/{RUN_NAME} --split test --figures {RUNS}/{RUN_NAME}/figures

## Compare all evaluated runs

Writes a table and chart for every run on Drive.

In [ ]:
!python scripts/compare_runs.py --runs {RUNS} --out {RUNS}/comparison

## TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS}/{RUN_NAME}/tb